In [0]:
# endpoint_utils
import requests
import re
from pyspark.sql.functions import udf, col, expr
from pyspark.sql.types import ArrayType, StringType

def fetch_wiki_with_fallback(page_name, regex_pattern):
    """
    Função principal de busca na Wikipedia.
    Lógica: Tenta a página exata; se falhar, faz uma busca por termos similares.
    """
    if not page_name: return []
    # A Wikipedia bloqueia requisições sem User-Agent claro. 
    # Se o script parar de retornar dados, verifique se este cabeçalho ainda é aceito.
    headers = {"User-Agent": "DatabricksDataPipeline/1.0 (contact: your@email.com)"}
    session = requests.Session()
    session.headers.update(headers)
    
    url = "https://en.wikipedia.org/w/api.php"
    found_links = []

    def get_links_from_page(target_page):
        """Sub-função que extrai links de uma página específica via Regex e Metadados."""
        params = {
            "action": "parse", "page": target_page, "format": "json", 
            "prop": "text|externallinks", "redirects": 1 # 'redirects: 1' resolve nomes antigos automaticamente
        }
        try:
            res = session.get(url, params=params, timeout=10).json()
            if "parse" not in res: return []
            
            # 1. Busca no HTML (Usa o padrão REGEX passado como parâmetro)
            html = res["parse"]["text"]["*"]
            links = re.findall(regex_pattern, html)
            
            # 2. Busca nos 'Links Externos' oficiais (metadados da página)
            ex_links = res["parse"].get("externallinks", [])
            for link in ex_links:
                match = re.search(regex_pattern, link)
                if match: links.append(match.group(1))
            
            return links
        except: 
            return [] # Evitamos travar o job se uma página falhar

    # PASSO 1: Tenta a página exata (Ex: 'Felipe_Neto')
    found_links.extend(get_links_from_page(page_name))

    # PASSO 2: FALLBACK (Se a página exata não existir ou não tiver o link)
    # Faz uma busca interna na Wiki pelos 3 resultados mais relevantes.
    if not found_links:
        search_params = {
            "action": "query", "list": "search", "format": "json",
            "srsearch": page_name.replace("_", " "), "srlimit": 3
        }
        try:
            search_res = session.get(url, params=search_params, timeout=10).json()
            candidates = [r["title"] for r in search_res.get("query", {}).get("search", [])]
            
            for candidate in candidates:
                found_links.extend(get_links_from_page(candidate))
                if found_links: break # Encontrou em um candidato? Para de buscar.
        except: pass

    # LIMPEZA: Remove duplicatas e ignora links de arquivos (archive.org) para evitar dados sujos
    unique_links = list(set([l for l in found_links if "archive" not in l.lower()]))
    return unique_links

# --- REGISTRO DA UDF ---
# Transforma a função Python em uma função que o Spark consegue rodar em paralelo nos nós.
fetch_social_udf = udf(fetch_wiki_with_fallback, ArrayType(StringType()))

def run_enhanced_extraction(source_table, target_table, regex, output_col):
    """
    Gerencia a execução da extração em larga escala e escolhe o melhor link encontrado.
    """
    # 1. EXECUÇÃO DISTRIBUÍDA
    # Aplica a UDF para cada página única de Wiki na base.
    df_raw = spark.table(source_table).select("wiki_page").distinct() \
                  .withColumn("candidates", fetch_social_udf(col("wiki_page"), expr(f"'{regex}'"))) \
                  .withColumn("exploded", expr("explode_outer(candidates)"))

    # 2. SELEÇÃO DO MELHOR MATCH
    # Se a busca retornou vários links, calculamos a 'distância' entre o nome da página e o link.
    # O Levenshtein garante que pegaremos o link que mais se parece com o nome do criador.
    df_raw.createOrReplaceTempView("v_social_candidates")
    
    query = f"""
        WITH ranked AS (
            SELECT 
                wiki_page, 
                exploded as {output_col},
                -- Score de similaridade: quanto menor, mais parecido
                levenshtein(lower(wiki_page), lower(exploded)) as score,
                -- Cria um ranking para pegar apenas o 1º lugar por página
                row_number() OVER (PARTITION BY wiki_page ORDER BY levenshtein(lower(wiki_page), lower(exploded)) ASC) as rnk
            FROM v_social_candidates
            WHERE exploded IS NOT NULL
        )
        SELECT wiki_page, {output_col} FROM ranked WHERE rnk = 1
    """
    
    df_final = spark.sql(query)
    
    # 3. SALVAMENTO (Overwrite Total)
    # Recria a tabela de destino com os novos links encontrados.
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    df_final.write.format("delta").saveAsTable(target_table)
    
    print(f"✅ Extração concluída com sucesso em {target_table}")